In [ ]:
!pip install -q aiohttp wasmtime nest_asyncio faster-whisper yt-dlp

In [ ]:
            # UPLOAD + SUBMIT — retry up to 3 times
            submitted = False
            for attempt in range(3):
                try:
                    async with session.put(
                        f"{COORDINATOR_URL}/blobs?assignment_token={tok}",
                        data=output, timeout=aiohttp.ClientTimeout(total=30)
                    ) as r:
                        up = await r.json()
                    output_hash = up.get("hash", output_hash)
                except Exception as e:
                    _log(f"Upload failed (attempt {attempt+1}/3): {e}")
                    await asyncio.sleep(1)
                    continue

                try:
                    async with session.post(
                        f"{COORDINATOR_URL}/worker/submit",
                        json={"type": "submit", "task_id": task["id"], "assignment_token": tok,
                              "result": {"output_hash": output_hash, "execution_metadata": {"exit_code": exit_code, "stderr": worker_stderr}}},
                        timeout=aiohttp.ClientTimeout(total=10)
                    ) as r:
                        result = await r.json()
                    accepted = result.get("accepted", False)
                    _log(f"Submit: accepted={accepted}")
                    if accepted:
                        TOTAL_COMPLETED += 1
                        _log(f"Total: {TOTAL_COMPLETED}")
                        submitted = True
                        break
                    _log(f"Submit rejected (attempt {attempt+1}/3)")
                except Exception as e:
                    _log(f"Submit failed (attempt {attempt+1}/3): {e}")

                await asyncio.sleep(1)

            if not submitted:
                _log("Submit failed after 3 attempts — task will be requeued by stale check")